# Final best combination model

This notebook computes the final test PR-AUC for the combination model from its saved component predictions.

The final model is a rank-based combination:
- keep the strongest **GraphAgg ET** head
- use a short **0.4 SIGN + 0.6 stack** middle block
- use **SIGN** for the remaining tail

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

PROJECT_DIR = Path('..')

OUT_DIR = PROJECT_DIR / 'outputs' / 'final_combination_model'
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRED_CSV = OUT_DIR / 'newsplit_32_37_42_val_test_predictions_with_oof_gate.csv'
OUT_JSON = OUT_DIR / 'final_combination_model_score.json'

In [2]:
df = pd.read_csv(PRED_CSV)
test_df = df[df['split'] == 'test'].copy().reset_index(drop=True)

# Build the middle blend explicitly
test_df['blend_sign_stack_p'] = (
    0.4 * test_df['sign_single_p'] +
    0.6 * test_df['stack_cb_p']
)

# Final combination structure on test
k1_test = 589   # GraphAgg head
k2_test = 64    # 1% middle blend block

def rank_scores_from_order(order_idx, n):
    scores = np.empty(n, dtype=float)
    scores[order_idx] = np.linspace(1.0, 0.0, n, endpoint=False)
    return scores

def hard_budget_gate_scores(df, head_col, mid_col, tail_col, k1, k2):
    n = len(df)
    idx_all = np.arange(n)

    # Head: top k1 by GraphAgg
    head_order = idx_all[np.argsort(-df[head_col].to_numpy(), kind='mergesort')]
    head_idx = head_order[:k1]
    head_set = set(head_idx.tolist())

    # Middle: next k2 by 0.4 SIGN + 0.6 stack among remaining rows
    rem1 = np.array([i for i in idx_all if i not in head_set], dtype=int)
    mid_order = rem1[np.argsort(-df.iloc[rem1][mid_col].to_numpy(), kind='mergesort')]
    mid_idx = mid_order[:k2]
    mid_set = set(mid_idx.tolist())

    # Tail: remaining rows by SIGN
    rem2 = np.array([i for i in rem1 if i not in mid_set], dtype=int)
    tail_order = rem2[np.argsort(-df.iloc[rem2][tail_col].to_numpy(), kind='mergesort')]

    final_order = np.concatenate([head_idx, mid_idx, tail_order])
    return rank_scores_from_order(final_order, n)

final_combo_score = hard_budget_gate_scores(
    test_df,
    head_col='graphagg_p',
    mid_col='blend_sign_stack_p',
    tail_col='sign_single_p',
    k1=k1_test,
    k2=k2_test,
)

final_test_pr_auc = float(average_precision_score(test_df['y'], final_combo_score))
final_test_roc_auc = float(roc_auc_score(test_df['y'], final_combo_score))

print("Best test PR-AUC:", round(final_test_pr_auc, 6))
print("Best test ROC-AUC:", round(final_test_roc_auc, 6))

Best test PR-AUC: 0.918653
Best test ROC-AUC: 0.97247


In [3]:
result = {
    'model': 'final_combination_model',
    'description': 'GraphAgg head -> 0.4 SIGN + 0.6 stack -> SIGN tail',
    'k1_test': k1_test,
    'k2_test': k2_test,
    'test_pr_auc': final_test_pr_auc,
    'test_roc_auc': final_test_roc_auc,
}

with open(OUT_JSON, 'w') as f:
    json.dump(result, f, indent=2)

print(json.dumps(result, indent=2))

{
  "model": "final_combination_model",
  "description": "GraphAgg head -> 0.4 SIGN + 0.6 stack -> SIGN tail",
  "k1_test": 589,
  "k2_test": 64,
  "test_pr_auc": 0.9186530169173815,
  "test_roc_auc": 0.9724702347727621
}
